In [2]:
import pandas as pd

In [3]:
# Open files up 
healthy_indices = pd.read_csv('data/is_healthy_food.csv', header=None)[0].to_list()
df_interactions = pd.read_csv('data/RAW_interactions.csv')

# Filter interactions for healthy recipes
is_healthy = df_interactions['recipe_id'].isin(healthy_indices)

df_healthy_interactions = df_interactions.loc[is_healthy, ['user_id', 'recipe_id', 'rating']]

# Create the utility matrix
df_utility = df_healthy_interactions.pivot(
    index='user_id',
    columns='recipe_id',
    values='rating'
)

In [7]:
import pandas as pd
from surprise import Dataset, Reader, KNNBasic
from surprise.model_selection import train_test_split
from surprise import accuracy


# -------------------------------
# 2. Prepare Surprise Dataset
# -------------------------------
reader = Reader(rating_scale=(0, 5))
data = Dataset.load_from_df(df_interactions[['user_id', 'recipe_id', 'rating']], reader)

# Split data into train/test
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# -------------------------------
# 3. Train KNN Model
# -------------------------------
sim_options = {
    'name': 'cosine',     # similarity metric
    'user_based': True    # user-based CF; False for item-based
}

model = KNNBasic(sim_options=sim_options)
model.fit(trainset)

# Evaluate on test set
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
print(f"RMSE on test set: {rmse}")

# -------------------------------
# 4. Prepare for Healthy Item Predictions
# -------------------------------
healthy_item_ids = set(healthy_indices)
user_ids = df_interactions['user_id'].unique()

# Generate predictions for healthy items only
all_preds = []
for uid in user_ids:
    for iid in healthy_item_ids:
        pred = model.predict(uid, iid)
        all_preds.append(pred)

# -------------------------------
# 5. Convert predictions to DataFrame (optional)
# -------------------------------
pred_df = pd.DataFrame([{
    'user_id': p.uid,
    'recipe_id': p.iid,
    'est_rating': p.est
} for p in all_preds])

# -------------------------------
# 6. Example: Top-5 healthy recommendations per user
# -------------------------------
top_n = pred_df.groupby('user_id').apply(
    lambda x: x.nlargest(5, 'est_rating')
).reset_index(drop=True)

print(top_n.head(10))


: 

In [6]:
user_counts = df_interactions.groupby('user_id')['rating'].count()
print(user_counts.describe())


count    226570.000000
mean          4.997868
std          49.663111
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        7671.000000
Name: rating, dtype: float64


In [7]:
item_counts = df_interactions.groupby('recipe_id')['rating'].count()
print(item_counts.describe())


count    231637.000000
mean          4.888541
std          17.532481
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max        1613.000000
Name: rating, dtype: float64


In [15]:
active_users = user_counts[user_counts >= 200].index
popular_items = item_counts[item_counts >= 200].index

df_filtered = df_interactions[
    df_interactions['user_id'].isin(active_users) &
    df_interactions['recipe_id'].isin(popular_items)
]


In [16]:
num_users = df_filtered['user_id'].nunique()
num_items = df_filtered['recipe_id'].nunique()
num_ratings = len(df_filtered)

sparsity = 1 - (num_ratings / (num_users * num_items))
print(f"Sparsity: {sparsity:.2%}")


Sparsity: 92.74%


In [17]:
df_filtered

,user_id,recipe_id,date,rating,review
6034,60989,30081,2003-07-10,5,"Oh my goodness! This was wonderful, yummy and ..."
6042,98919,30081,2003-09-21,5,Tried your Gyros today and thought they were o...
6046,134624,30081,2004-04-29,5,Oh yes! I love gyros and haven't had them in ...
6048,46660,30081,2004-06-02,5,These were very good! I made the ground beef ...
6050,58038,30081,2004-08-28,4,Nice change of pace burger with good taste. I...
...,...,...,...,...,...
1129789,789516,43072,2011-04-13,0,easy! doubled the recipe and used 2 whole eggs.
1129795,81611,43072,2011-07-19,5,"I tripled this, using 1 1/2 c oatmeal and cott..."
1129812,2324285,43072,2012-07-22,5,These taste great. I added a teaspoon of agav...
1129815,540346,43072,2012-11-07,4,"Since this said it only made one serving, I qu..."
